# RE-THREAD: Garment Damage Segmentation Training (YOLOv8-Seg)

**Team Circular Transformation · PCCOE Pune**  
**Objective:** Train an Ultralytics YOLOv8 small segmentation model (`yolov8n-seg.pt`) to locate garment defects (tears, holes, stains) and compute the damaged area percentage.

---

### How this notebook works (Beginner Friendly Guide):
1. **Check GPU:** Ensure Google Colab GPU is active (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. **Mount Google Drive:** So your trained `best.pt` weights file is saved safely.
3. **Install Ultralytics & Roboflow:** The computer vision libraries.
4. **Download Dataset:** Exported in **YOLOv8** format (containing `data.yaml`).
5. **Train YOLOv8-seg:** Fine-tune pretrained `yolov8n-seg.pt` on the defect dataset.
6. **Validate:** Inspect mAP metrics and segmentation masks.
7. **Save Weights:** Copy `runs/segment/train/weights/best.pt` to Google Drive as `damage.pt`.
8. **Download to Backend:** Place `damage.pt` into `re-thread/weights/damage.pt`.

## Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Please click Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Mount Google Drive
This ensures your trained weights (`best.pt`) are not lost if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create an output directory in Google Drive for RE-THREAD weights
OUTPUT_DIR = '/content/drive/MyDrive/rethread_weights'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Weights will be saved to: {OUTPUT_DIR}")

## Step 3: Install Ultralytics YOLOv8 & Roboflow

In [ ]:
!pip install --upgrade ultralytics roboflow

import ultralytics
ultralytics.checks()

## Step 4: Download Roboflow Fabric/Garment Defect Dataset

Replace the code below with your Roboflow project snippet (exported in **YOLOv8** format):
- Dataset recommendations: FabricDefect, Clothing-Defects, or FabricSpotDefect from Roboflow Universe.
- Export format: **YOLOv8**.

In [ ]:
# Option A: Using the Roboflow Python SDK (Recommended)
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("fabric-defects").project("clothing-damage-seg")
# version = project.version(1)
# dataset = version.download("yolov8")
# DATA_YAML = f"{dataset.location}/data.yaml"

# Option B: Direct zip download or sample dataset creation for demonstration
import os

# If you downloaded a zip from Roboflow or Kaggle, unzip it here:
# !unzip -q /content/clothing_damage_yolov8.zip -d /content/dataset
# DATA_YAML = "/content/dataset/data.yaml"

# Verify that data.yaml exists
DATA_YAML = "/content/dataset/data.yaml"
if not os.path.exists(DATA_YAML):
    print(f"Please set DATA_YAML to the path of your downloaded data.yaml file.")
else:
    print(f"Found data.yaml at: {DATA_YAML}")

## Step 5: Train YOLOv8-Seg on the Defect Dataset

We start with `yolov8n-seg.pt` (nano segmentation model, pretrained on COCO).
- `imgsz=640`: standard resolution for garment details
- `epochs=50`: good balance of accuracy and training speed
- `batch=16`: optimal for Google Colab T4 GPU (adjust to 8 if out-of-memory occurs)

In [ ]:
from ultralytics import YOLO

# 1. Load pretrained YOLOv8-seg nano model
model = YOLO("yolov8n-seg.pt")

# 2. Train on defect dataset
# Note: Change 'data' to your actual data.yaml path
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="rethread_training",
    name="damage_yolov8_seg",
    save=True,
)

print("Training complete!")

## Step 6: Validate the Trained Model

In [ ]:
# Load best weights from the training run
best_weights_path = "rethread_training/damage_yolov8_seg/weights/best.pt"
trained_model = YOLO(best_weights_path)

# Evaluate metrics on validation split
metrics = trained_model.val()
print(f"Box mAP50-95: {metrics.box.map:.4f}")
if hasattr(metrics, 'seg'):
    print(f"Mask mAP50-95: {metrics.seg.map:.4f}")

## Step 7: Save Trained Weights to Google Drive
We save the file as `damage.pt` so it is ready to drop into the `re-thread/weights/` directory.

In [ ]:
import shutil

source_weights = "rethread_training/damage_yolov8_seg/weights/best.pt"
destination_weights = f"{OUTPUT_DIR}/damage.pt"

shutil.copy(source_weights, destination_weights)
print(f"Successfully saved trained model to Google Drive at:\n{destination_weights}")

# Step to integrate with the RE-THREAD backend:
print("\n--- NEXT STEPS FOR LOCAL BACKEND INTEGRATION ---")
print("1. Download damage.pt from your Google Drive (rethread_weights/damage.pt).")
print("2. Move it to: re-thread/weights/damage.pt")
print("3. Restart the FastAPI backend. It will automatically load the real model at startup!")

## Step 8: Test Single-Image Inference and Area % Calculation
This mirrors the exact Python logic implemented in `backend/app/ml/damage.py`.

In [ ]:
import numpy as np

# Test on a validation image
# results = trained_model.predict(source="/content/test_tear.jpg", conf=0.25)
# for r in results:
#     has_damage = len(r.boxes) > 0
#     if r.masks is not None and len(r.masks) > 0:
#         # Sum of mask pixels / total image pixels
#         mask_area = np.sum(r.masks.data.cpu().numpy() > 0.5)
#         total_area = r.orig_shape[0] * r.orig_shape[1]
#         area_pct = round((mask_area / total_area) * 100.0, 2)
#     else:
#         area_pct = 0.0
#     print(f"Detected: {has_damage}, Damaged Area: {area_pct}%")